# Thesis figures notebook

Notebook nay la guided thesis-analysis notebook cho bo hinh benchmark. Moi figure co:

- heading rieng;
- block giai thich muc dich, du lieu, cach doc va diem can quan sat;
- code cell sinh dung figure do;
- mot ghi chu ngan ve cach dien giai.

Tai lieu dien giai chi tiet nhat nam tai:

`docs/thesis_figure_analysis.md`

Notebook nay dung de tai sinh hinh va doc nhanh theo narrative: calibration -> S1 -> S2 -> repeatability -> S3.


## Setup va load data

Cell nay tim repo root, nap cac CSV analysis, import plotting pipeline tu `scripts/generate_charts.py`, va dinh nghia helper hien thi anh vua sinh.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "results_analysis" / "comparison_AB.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root")


REPO_ROOT = find_repo_root()
ANALYSIS_DIR = REPO_ROOT / "results_analysis"
OUT_DIR = REPO_ROOT / "docs" / "figures" / "thesis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / "scripts"))
import generate_charts as charts

aggregated, comparison = charts.load_csvs(ANALYSIS_DIR)
repeatability_raw = charts.repeatability_raw_p99(REPO_ROOT)
repeatability_summary = charts.repeatability_summary(repeatability_raw)
calibration_selection_path = ANALYSIS_DIR / "calibration_load_selection.csv"
calibration_selection = pd.read_csv(calibration_selection_path) if calibration_selection_path.exists() else pd.DataFrame()


def show_fig(name: str) -> None:
    path = OUT_DIR / f"{name}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    display(Image(filename=str(path)))


print("repo:", REPO_ROOT)
print("aggregated:", aggregated.shape)
print("comparison:", comparison.shape)
print("repeatability raw:", repeatability_raw.shape)


## Inspect input tables

Buoc nay giup kiem tra notebook dang doc dung outputs hien tai. Neu schema thay doi, nen dung lai truoc khi dien giai hinh.


In [ ]:
print("comparison_AB columns:")
print(list(comparison.columns))

print("\naggregated_summary columns:")
print(list(aggregated.columns))

if not calibration_selection.empty:
    print("\ncalibration_load_selection:")
    display(calibration_selection)

print("\nrepeatability summary head:")
display(repeatability_summary.head())


## Figure: fig-calibration-curve

### Muc dich

Figure nay giai thich vi sao benchmark chon L1/L2/L3. No khong so sanh Mode A voi Mode B, ma the hien p99 latency theo target QPS trong calibration sweep.

### Du lieu va cach doc

- Du lieu: newest non-pilot calibration CSV trong `results/calibration/`.
- Truc x: target QPS.
- Truc y: p99 latency (ms).
- Cac vach L1/L2/L3 lien ket calibration voi benchmark final.

### Diem can quan sat

- L1 = 100 QPS co p99 thap.
- L2 = 400 QPS co tail latency ro hon nhung error rate van 0%.
- L3 final = 800 QPS vi day la diem p99 tang manh dau tien, nhung achieved RPS van bam target.
- 1600 QPS ton tai trong calibration nhung khong phai L3 final; khong goi 800 QPS la saturation point tuyet doi.


In [ ]:
charts.chart_calibration_curve(REPO_ROOT, OUT_DIR)
show_fig("fig-calibration-curve")


## Figure: fig-s1-p99-vs-load

### Muc dich

Figure nay tap trung vao tail latency cua S1. S1 la steady-state workload, nen line chart theo load L1/L2/L3 la cach doc tu nhien.

### Du lieu va cach doc

- Du lieu: `comparison_AB.csv`, scenario S1, metric `p99_ms`.
- Truc x: load level.
- Truc y: p99 latency (ms).
- Hai line: Mode A va Mode B.

### Diem can quan sat

Mode B co mean p99 thap hon o L1/L2/L3. L2 va L3 duoc danh dau significant trong CSV; L1 p99 chi nen doc la xu huong.


In [ ]:
charts.chart_s1_p99_vs_load(comparison, OUT_DIR)
show_fig("fig-s1-p99-vs-load")


## Figure: fig-s1-latency-small-multiples

### Muc dich

Figure nay tach p50, p90 va p99 thanh cac panel rieng de xem xu huong co nhat quan tren nhieu phan cua latency distribution hay khong.

### Du lieu va cach doc

- Du lieu: `comparison_AB.csv`, scenario S1.
- Metrics: `p50_ms`, `p90_ms`, `p99_ms`.
- Moi panel so sanh Mode A voi Mode B theo L1/L2/L3.

### Diem can quan sat

Mode B nam thap hon Mode A tren p50/p90/p99 ve mean. O L3, khoang cach ro hon, dac biet p90 va p99. Van can doi chieu significance rieng tung metric/load.


In [ ]:
charts.chart_s1_latency_small_multiples(comparison, OUT_DIR)
show_fig("fig-s1-latency-small-multiples")


## Figure: fig-s1-delta-p99

### Muc dich

Figure nay rut gon S1 p99 thanh delta % cua Mode B so voi Mode A. Day la hinh de dung cho slide vi doc huong cai thien rat nhanh.

### Du lieu va cach doc

- Du lieu: `comparison_AB.csv`, scenario S1, metric `p99_ms`.
- Truc x: L1/L2/L3.
- Truc y: delta %.
- Gia tri am nghia la Mode B co p99 thap hon Mode A.
- Dau `*` nghia la `significant == YES`.

### Diem can quan sat

Tat ca cot deu am, nhung L1 khong significant. Ket luan manh nen tap trung vao L2 va L3.


In [ ]:
charts.chart_s1_delta_p99(comparison, OUT_DIR)
show_fig("fig-s1-delta-p99")


## Figure: fig-s2-phase-p99-l2

### Muc dich

Figure nay phan tich S2 L2 theo tung phase. S2 co ramp-up, sustained, burst va cooldown, nen khong duoc gop thanh mot bucket duy nhat.

### Du lieu va cach doc

- Du lieu: `comparison_AB.csv`, scenario S2, load L2, metric `p99_ms`.
- Truc x: S2 phase.
- Truc y: p99 latency (ms).
- Hai line: Mode A va Mode B.

### Phase can chu y

- Ramp-up: Mode B thap hon va significant.
- Sustained: Mode B thap hon ve mean nhung khong significant.
- Burst 1: Mode B cao hon Mode A; day la ngoai le can neu ro.
- Burst 2: khac biet gan nhu khong dang ke.
- Burst 3: Mode B thap hon ve mean nhung khong significant.
- Cooldown: Mode B thap hon nhe.


In [ ]:
charts.chart_s2_phase_p99(comparison, OUT_DIR, "L2")
show_fig("fig-s2-phase-p99-l2")


## Figure: fig-s2-phase-p99-l3

### Muc dich

Figure nay phan tich S2 tai load L3, noi workload nang hon va cac burst phase co the lam ro khac biet tail latency.

### Du lieu va cach doc

- Du lieu: `comparison_AB.csv`, scenario S2, load L3, metric `p99_ms`.
- Truc x: S2 phase.
- Truc y: p99 latency (ms).
- Hai line: Mode A va Mode B.

### Phase can chu y

Mode B co mean p99 thap hon Mode A o tat ca phase L3. Burst 1, Burst 2, Burst 3 va Cooldown co delta am lon hon. Tuy nhien cac phase L3 khong duoc danh dau significant, nen viet la xu huong mean, khong phai ket luan thong ke tuyet doi.


In [ ]:
charts.chart_s2_phase_p99(comparison, OUT_DIR, "L3")
show_fig("fig-s2-phase-p99-l3")


## Figure: fig-s2-p99-delta-heatmap

### Muc dich

Heatmap nay tra loi nhanh: Mode B tot hon hay kem hon Mode A o phase nao va load nao trong S2.

### Du lieu va cach doc

- Du lieu: raw final `fortio_phase*.json` cua S2.
- Hang: S2 phase.
- Cot: L2 va L3.
- O: delta % cua p99 Mode B so voi Mode A.
- Am = Mode B p99 thap hon; duong = Mode B p99 cao hon.

### Diem can quan sat

L3 co tat ca o am, dac biet burst/cooldown. L2 Burst 1 la ngoai le duong +12.1%. L2 Burst 2 gan nhu khong khac biet. Khong doc mau heatmap nhu bang chung significance.


In [ ]:
charts.chart_s2_p99_delta_heatmap(REPO_ROOT, OUT_DIR)
show_fig("fig-s2-p99-delta-heatmap")


## Figure: fig-repeatability-p99-dotplot

### Muc dich

Figure nay cho thay do on dinh cua benchmark qua cac raw run. No bo sung cho cac hinh mean/CI bang cach cho thay tung run nam o dau.

### Du lieu va cach doc

- Du lieu: raw final Fortio JSON.
- Moi dot: mot raw final run.
- Vach ngang ngan: mean cua group.
- Khong dung boxplot vi moi group chi co n=3.

### Diem can quan sat

S1 L3 Mode B va S3 L3 Policy OFF/ON co dot gan nhau, CV thap. S2 co bien dong cao hon, dac biet burst/cooldown. Dieu nay giai thich vi sao S2 can phase-aware va can wording than trong.


In [ ]:
charts.write_repeatability_summary(repeatability_raw, ANALYSIS_DIR)
charts.chart_repeatability_p99_dotplot(repeatability_raw, OUT_DIR)
display(repeatability_summary.sort_values("cv_pct", ascending=False).head(10))
show_fig("fig-repeatability-p99-dotplot")


## Figure: fig-s3-policy-overhead-p99

### Muc dich

Figure nay la hinh chinh cua S3. No do policy overhead bang cach so sanh Mode B policy OFF va policy ON, khong phai Mode A vs Mode B.

### Du lieu va cach doc

- Du lieu: `aggregated_summary.csv`, S3 final groups.
- Truc x: load L2/L3.
- Truc y: p99 latency (ms).
- Marker: Policy OFF va Policy ON.

### Diem can quan sat

L2 ON thap hon OFF nhe; L3 ON cao hon OFF nhe. CI chong lan va delta nho, nen ket luan an toan la policy don gian khong tao overhead p99 dang ke trong dataset nay.


In [ ]:
charts.chart_s3_policy_p99(aggregated, OUT_DIR)
show_fig("fig-s3-policy-overhead-p99")


## Figure: fig-s3-policy-overhead-rps

### Muc dich

Figure nay la supporting evidence cho S3: kiem tra policy ON co lam giam achieved RPS khong.

### Du lieu va cach doc

- Du lieu: `aggregated_summary.csv`, S3 final groups.
- Panel rieng cho L2 va L3 vi OFF/ON gan nhu trung nhau.
- Truc y: achieved RPS.
- Target line cho thay RPS co bam target hay khong.

### Diem can quan sat

RPS gan nhu khong doi: L2 khoang 399.992, L3 khoang 799.974 vs 799.965. Error rate 0%. Hinh nay nen dung nhu bang chung phu; p99 van la hinh chinh cho overhead.


In [ ]:
charts.chart_s3_policy_rps(aggregated, OUT_DIR)
show_fig("fig-s3-policy-overhead-rps")


## Ket luan doc notebook

Notebook nay tao lai tung figure theo dung thu tu thesis narrative. Khi viet report, dung `docs/thesis_figure_analysis.md` de lay dien giai chi tiet va wording thesis-safe.

Checklist tranh overclaim:

- L3 la high-stress/tail-spike load, khong phai saturation point tuyet doi.
- S1 co bang chung tot hon o L2/L3; L1 p99 la xu huong.
- S2 phai phase-aware; L2 Burst 1 la ngoai le can neu ro.
- Repeatability voi n=3 la mo ta dataset, khong phai uoc luong thong ke day du.
- S3 la policy OFF vs ON trong Mode B; khong claim Hubble DROPPED neu artifact khong ho tro.
